In [ ]:
import cv2
import numpy as np
import tensorflow as tf

# ----------------------------
# CONFIG
# ----------------------------
MODEL_PATH = "cnn_lstm_deepfake_model.h5"   # your saved model
VIDEO_PATH = "test_video.mp4"               # video to test
IMG_SIZE = 224
MAX_FRAMES = 20

# ----------------------------
# Load Model
# ----------------------------
print("Loading model...")
model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully.\n")

# ----------------------------
# Function: Load & preprocess video
# ----------------------------
def load_video(path, max_frames=MAX_FRAMES, img_size=IMG_SIZE):
    cap = cv2.VideoCapture(path)
    
    frames = []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    if total > 0:
        # Sample frames evenly
        indices = np.linspace(0, total - 1, num=max_frames, dtype=int)
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (img_size, img_size))
            frames.append(frame)
    else:
        # fallback: sequential read
        while len(frames) < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (img_size, img_size))
            frames.append(frame)

    cap.release()

    # Handle fewer frames by repeating the last one
    if len(frames) == 0:
        # Return zeros if video cannot be read
        arr = np.zeros((max_frames, img_size, img_size, 3), dtype=np.float32)
        return arr

    if len(frames) < max_frames:
        last = frames[-1]
        while len(frames) < max_frames:
            frames.append(last)

    arr = np.array(frames).astype("float32") / 255.0
    return arr

# ----------------------------
# Predict
# ----------------------------
print("Processing video...\n")
video_array = load_video(VIDEO_PATH)

# Add batch dimension (1, max_frames, 224, 224, 3)
video_array = np.expand_dims(video_array, axis=0)

print("Running prediction...\n")
pred = model.predict(video_array)[0][0]

confidence = float(pred)
label = "FAKE" if confidence > 0.5 else "REAL"

print("==================================")
print("Prediction Result:")
print("Video:", VIDEO_PATH)
print("Label:", label)
print("Confidence:", round(confidence * 100, 2), "%")
print("==================================\n")
